# 模块二：SwellEx-96 GPU 网络训练、预测与实验结果分析

本 Notebook 专为 **Google Colab GPU 运行时**（`Runtime -> Change runtime type -> T4 / L4 / A100 GPU`）设计，直接消费 **模块一（`01_data_processing_cpu.ipynb`，基于 Google Drive 文件夹 [`1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj`](https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link) 生成）** 预先保存在 `/content/drive/MyDrive/SwellEx_Colab/datasets/` 中的 HDF5 数据集，**无需再次读取 `Origindata/`，也不会重复执行任何 CPU 频域特征提取**。

### 高效 I/O 与持久化策略
1. **自动引导仓库**：Colab 会话启动后先挂载 Google Drive，若实例中尚未拉取仓库则自动克隆 `SwellEx` 到 `/content/SwellEx`。
2. **读数据走本地 NVMe 缓存**：根据 Google Drive 上的 `dataset_manifest.json`，按需仅把当前实验用到的 HDF5 目录复制到 Colab 本地高速盘 `/content/local_datasets/`，避免多轮（100 Epochs）`DataLoader` 反复穿透 Google Drive FUSE 造成 GPU 空转等待。
3. **写结果直存 Google Drive**：模型权重（`best.pt`）、训练日志、测试集预测 CSV 与评估图表直接写入 `/content/drive/MyDrive/SwellEx_Colab/experiments/<run_name>/`，即使 Colab 会话断开也不会丢失成果。

In [1]:
# Cell 1: Google Drive 挂载、仓库自动引导、GPU 环境检查与 dataset_manifest.json 读取
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

# --- 与模块一保持一致的 Google Drive 工作区配置 ---
DRIVE_WORKSPACE_SUBDIR = "SwellEx_Colab"  # 对应 /content/drive/MyDrive/SwellEx_Colab
ORIGINDATA_DRIVE_URL = "https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link"
REPO_GIT_URL = "https://github.com/AmazingUmi/SwellEx.git"
# ------------------------------------------------

IN_COLAB = (
    "google.colab" in sys.modules
    or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)
)

DRIVE_MOUNT_ROOT = Path("/content/drive")
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)


def ensure_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path("/content/SwellEx")]
    if IN_COLAB and DRIVE_MOUNT_ROOT.exists():
        candidates.extend([
            DRIVE_MOUNT_ROOT / "MyDrive" / "SwellEx",
            DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR / "SwellEx",
        ])
    for candidate in candidates:
        if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
            return candidate.resolve()
    if IN_COLAB:
        clone_dst = Path("/content/SwellEx")
        print(f"📥 未检测到本地仓库，正在克隆 {REPO_GIT_URL} -> {clone_dst} ...")
        proc = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_GIT_URL, str(clone_dst)],
            capture_output=True,
            text=True,
        )
        if proc.returncode == 0 and (clone_dst / "scripts" / "remote_workflow" / "packs.py").exists():
            return clone_dst.resolve()
        raise FileNotFoundError(
            f"自动克隆仓库失败（可能仓库为私有）: {proc.stderr.strip()}\n"
            "请先将 SwellEx 仓库上传至 /content/drive/MyDrive/SwellEx 或在 Colab 执行 git clone 后重试。"
        )
    raise FileNotFoundError("未找到项目根目录（缺少 scripts/remote_workflow/packs.py）。")


PROJECT_ROOT = ensure_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IN_COLAB:
    DRIVE_BASE = DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR
    LOCAL_DATASET_CACHE = Path("/content/local_datasets")
else:
    DRIVE_BASE = Path(os.environ.get("SWELLEX_DRIVE_SIM", PROJECT_ROOT / "outputs" / "drive_staging"))
    LOCAL_DATASET_CACHE = PROJECT_ROOT / "outputs" / "local_dataset_cache"

DRIVE_DATASET_ROOT = DRIVE_BASE / "datasets"
DRIVE_EXP_ROOT = DRIVE_BASE / "experiments"
MANIFEST_PATH = DRIVE_DATASET_ROOT / "dataset_manifest.json"

HAS_TORCH = importlib.util.find_spec("torch") is not None
CUDA_AVAILABLE = False
if HAS_TORCH:
    import torch
    CUDA_AVAILABLE = bool(torch.cuda.is_available())
    if CUDA_AVAILABLE:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ CUDA GPU 已就绪: {gpu_name} ({vram_gb:.1f} GiB), PyTorch {torch.__version__}")
    else:
        print("⚠️ 未检测到可用 CUDA GPU！在 Colab 中请切换 Runtime -> Change runtime type -> GPU。")
else:
    print("ℹ️ 当前本地解释器未安装 PyTorch（将以配置/历史结果分析模式运行；Colab GPU 环境自带 PyTorch）。")

MANIFEST = (
    json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if MANIFEST_PATH.exists()
    else {"schema_version": 1, "presets": {}, "moe_learn": {}}
)
origin_prov = MANIFEST.get("origindata_source", {})
print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"MANIFEST_PATH      : {MANIFEST_PATH} (存在={MANIFEST_PATH.exists()})")
if origin_prov:
    print(f"原始数据溯源 (只读): FolderID={origin_prov.get('drive_folder_id')} ({origin_prov.get('resolution_mode')})")
elif not MANIFEST_PATH.exists():
    print(f"ℹ️ 尚未发现 dataset_manifest.json：请先在 CPU 运行时执行 01_data_processing_cpu.ipynb（已绑定 {ORIGINDATA_DRIVE_URL}）。")
print(f"可用预设数据集     : {list(MANIFEST.get('presets', {}).keys())}")
print(f"可用 MoE_learn 集  : {list(MANIFEST.get('moe_learn', {}).keys())}")
print(f"实验结果保存根目录 : {DRIVE_EXP_ROOT}")

ℹ️ 当前本地解释器未安装 PyTorch（将以配置/历史结果分析模式运行；Colab GPU 环境自带 PyTorch）。
PROJECT_ROOT       : /Volumes/exDateDisk/projects/SwellEx
MANIFEST_PATH      : /Volumes/exDateDisk/projects/SwellEx/outputs/drive_staging/datasets/dataset_manifest.json (存在=False)
ℹ️ 尚未发现 dataset_manifest.json：请先在 CPU 运行时执行 01_data_processing_cpu.ipynb（已绑定 https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link）。
可用预设数据集     : []
可用 MoE_learn 集  : []
实验结果保存根目录 : /Volumes/exDateDisk/projects/SwellEx/outputs/drive_staging/experiments

## 1. 五个实验包与容量/专家数预算速览

下方单元格直接调用仓库纯标准库模块（`scripts.remote_workflow.packs`、容量缩放规则与残差 MoE 专家选择规则），帮助在启动 GPU 训练前确认合法的 `PACK`、`METHOD`、`NETWORK` 与样本量档位。

In [2]:
# Cell 2: 查看五个实验包支持的路由/网络组合，以及按训练样本数缩放的容量与专家预算
from scripts.remote_workflow.packs import PACKS
from scripts.network_training.data_scale_pack.common.scaling import (
    DEFAULT_SCALING_POLICY as CNN_POLICY,
    resolve_capacity as resolve_cnn_capacity,
)
from scripts.network_training.MoE_data_scale.common.scaling import (
    DEFAULT_SCALING_POLICY as MOE_POLICY,
    resolve_capacity as resolve_moe_capacity,
)

print("=== 实验包注册表 (scripts/remote_workflow/packs.py) ===")
for name, pack in PACKS.items():
    fusion_note = f" | fusions={list(pack.fusion_modes)}" if pack.fusion_modes else ""
    print(f"- {name:18s}: methods={list(pack.methods)} | networks={list(pack.networks)}{fusion_note}")

print("\n=== 训练样本数 N -> CNN 深度档位 (data_scale_pack) vs MoE 专家数 (MoE_data_scale) ===")
print(f"{'N_train':>8s} | {'CNN Profile':>10s} | {'Repeats':>12s} | {'MoE Experts':>11s} | {'Top-k':>5s}")
print("-" * 60)
for n_rows in (80, 150, 336, 480, 900, 1500):
    cnn_cap = resolve_cnn_capacity("auto", train_count=n_rows, base_channels=16, scaling_policy=CNN_POLICY)
    moe_cap = resolve_moe_capacity("auto", train_count=n_rows, base_channels=16, scaling_policy=MOE_POLICY)
    print(
        f"{n_rows:8d} | {cnn_cap['selected_profile']:>10s} | "
        f"{str(cnn_cap['stage_repeats']):>12s} | "
        f"{'E' + str(moe_cap['num_experts']):>11s} | {moe_cap['top_k']:5d}"
    )

=== 实验包注册表 (scripts/remote_workflow/packs.py) ===
- base_pack         : methods=['RBD', 'ELM', 'SCM'] | networks=['complex_CNN', 'real_CNN', 'ResNet18', 'ResNet50', 'GRNN']
- joint_feature_pack: methods=['RBD', 'ELM', 'SCM', 'RBD_ELM', 'RBD_SCM', 'ELM_SCM', 'RBD_ELM_SCM'] | networks=['complex_CNN', 'real_CNN', 'ResNet18', 'ResNet50', 'GRNN'] | fusions=['raw_concat', 'block_rms']
- data_scale_pack   : methods=['RBD', 'ELM', 'SCM'] | networks=['complex_CNN', 'complex_CNN_small', 'complex_CNN_medium', 'complex_CNN_large', 'complex_CNN_xlarge', 'complex_CNN_baseline']
- MoE_data_scale    : methods=['RBD', 'ELM', 'SCM', 'RBD_SCM'] | networks=['complex_MoE', 'complex_MoE_e1', 'complex_MoE_e2', 'complex_MoE_e4', 'complex_MoE_e8']
- MoE_learn         : methods=['RBD_SCM'] | networks=['complex_MoE_learn']

=== 训练样本数 N -> CNN 深度档位 (data_scale_pack) vs MoE 专家数 (MoE_data_scale) ===
 N_train | CNN Profile |      Repeats | MoE Experts | Top-k
---------------------------------------------------------

## 2. 标准神经网络与独立 GRNN 训练 / 预测运行器（GPU）

支持 `base_pack`、`joint_feature_pack`、`data_scale_pack` 与 `MoE_data_scale`：
1. 自动从 `DRIVE_DATASET_ROOT` 将目标方法对应的 HDF5 目录复制到本地高速缓存 `LOCAL_DATASET_CACHE`；
2. 神经网络依次执行 `train`（保存最优验证集 `best.pt`）与 `predict`（输出测试集 `MAE / RMSE / R²` 与逐样本预测 CSV）；独立 `GRNN` 依次执行 `build`（5 折交叉验证选 `sigma`）与 `predict`；
3. 所有模型权重、历史记录和预测结果直接写入 Google Drive 的 `DRIVE_EXP_ROOT/<run_name>/`。

In [3]:
# Cell 3: 标准包 GPU 训练与测试预测（按需修改参数并将 RUN_TRAINING 设为 True）
from datetime import datetime
import subprocess

# --- 用户训练参数 ---
RUN_TRAINING = False               # 在 Colab GPU 就绪且模块一已生成数据后设为 True
DATASET_TAG = "train_before"       # 对应模块一写入 dataset_manifest.json 的 dataset_tag
PACK = "base_pack"                 # 可选: "base_pack" | "joint_feature_pack" | "data_scale_pack" | "MoE_data_scale"
METHOD = "SCM"                     # 如 "RBD", "ELM", "SCM", "RBD_SCM", "RBD_ELM_SCM"
NETWORK = "ResNet18"               # 如 "complex_CNN", "real_CNN", "ResNet18", "ResNet50", "GRNN",
                                   #    "complex_CNN_small", "complex_MoE_e2" 等
FUSION = "block_rms"               # 仅 joint_feature_pack 使用: "block_rms" | "raw_concat"
EPOCHS = 100
BATCH_SIZE = 64
TRAIN_FRACTION = 1.0               # data_scale_pack / MoE_data_scale 可设为 0.25, 0.5, 1.0 等做数据规模对比
SEED = 2026
# --------------------


def stage_h5_from_drive(pack: str, dataset_tag: str, method: str, fusion: str = "block_rms") -> Path:
    preset_info = MANIFEST.get("presets", {}).get(dataset_tag)
    if not preset_info:
        raise KeyError(
            f"dataset_manifest.json 中未找到 dataset_tag={dataset_tag!r}，请先在模块一生成并同步该预设。"
        )
    if pack in ("base_pack", "data_scale_pack") or (pack == "MoE_data_scale" and method != "RBD_SCM"):
        entry = preset_info["base_routes"][method]
    elif pack == "joint_feature_pack":
        entry = preset_info["joint_feature_pack"][fusion][method]
    elif pack == "MoE_data_scale" and method == "RBD_SCM":
        entry = preset_info["MoE_data_scale"]["RBD_SCM"]
    else:
        raise ValueError(f"不支持的包/方法组合: pack={pack}, method={method}")

    rel_dir = Path(entry["rel_dir"])
    src_dir = DRIVE_DATASET_ROOT / rel_dir
    dst_dir = LOCAL_DATASET_CACHE / rel_dir
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
    return dst_dir


def resolve_model_cli(pack: str, method: str, network: str) -> tuple[Path, str, list[str]]:
    m_low = method.lower()
    net_suffix_map = {
        "complex_CNN": "complex_cnn_range",
        "real_CNN": "real_cnn_range",
        "ResNet18": "resnet18_range",
        "ResNet50": "resnet50_range",
        "complex_CNN_small": "complex_cnn_small_range",
        "complex_CNN_medium": "complex_cnn_medium_range",
        "complex_CNN_large": "complex_cnn_large_range",
        "complex_CNN_xlarge": "complex_cnn_xlarge_range",
        "complex_CNN_baseline": "complex_cnn_baseline_range",
        "complex_MoE": "complex_moe_range",
        "complex_MoE_e1": "complex_moe_e1_range",
        "complex_MoE_e2": "complex_moe_e2_range",
        "complex_MoE_e4": "complex_moe_e4_range",
        "complex_MoE_e8": "complex_moe_e8_range",
    }
    if network == "GRNN":
        if pack not in ("base_pack", "joint_feature_pack"):
            raise ValueError(f"{pack} 不包含独立 GRNN。")
        entry = PROJECT_ROOT / f"scripts/network_training/{pack}/grnn/main.py"
        return entry, "GRNN", ["--method", method]

    suffix = net_suffix_map[network]
    if pack == "joint_feature_pack":
        entry = PROJECT_ROOT / "scripts/network_training/joint_feature_pack/joint/main.py"
        return entry, f"joint_{suffix}", ["--method", method]
    entry = PROJECT_ROOT / f"scripts/network_training/{pack}/{m_low}/main.py"
    return entry, f"{m_low}_{suffix}", []


entry_script, model_id, prefix_args = resolve_model_cli(PACK, METHOD, NETWORK)
print(f"目标入口脚本 : {entry_script.relative_to(PROJECT_ROOT)}")
print(f"解析模型标识 : {model_id}")

if not RUN_TRAINING:
    print("ℹ️ 当前 RUN_TRAINING=False（预览命令模式）。在 Colab GPU 准备好后将 RUN_TRAINING 设为 True 即可启动训练。")
else:
    if not CUDA_AVAILABLE:
        raise RuntimeError("拒绝静默回退到 CPU：请先将 Colab 运行时切换为 GPU！")
    local_data_dir = stage_h5_from_drive(PACK, DATASET_TAG, METHOD, FUSION)
    stamp = datetime.now().strftime("%m%d_%H%M%S")
    exp_dir = DRIVE_EXP_ROOT / f"{PACK}_{DATASET_TAG}_{METHOD}_{NETWORK}_{stamp}"
    exp_dir.mkdir(parents=True, exist_ok=True)
    print(f"📂 本地 SSD 数据缓存 : {local_data_dir}")
    print(f"💾 Google Drive 输出 : {exp_dir}")

    if NETWORK == "GRNN":
        cmds = [
            [sys.executable, "-B", str(entry_script), "build", *prefix_args,
             "--data", str(local_data_dir), "--device", "cuda",
             "--cv-folds", "5", "--batch-size", "1024", "--seed", str(SEED),
             "--output-dir", str(exp_dir)],
            [sys.executable, "-B", str(entry_script), "predict", *prefix_args,
             "--data", str(local_data_dir), "--device", "cuda",
             "--batch-size", "1024", "--output-dir", str(exp_dir)],
        ]
    else:
        extra_scale_args: list[str] = []
        if PACK == "data_scale_pack":
            extra_scale_args = ["--train-fraction", str(TRAIN_FRACTION), "--scale-reference-samples", "600", "--scale-exponent", "0.5"]
        elif PACK == "MoE_data_scale":
            extra_scale_args = ["--train-fraction", str(TRAIN_FRACTION), "--balance-loss-weight", "0.01"]

        cmds = [
            [sys.executable, "-B", str(entry_script), *prefix_args, "train",
             "--model", model_id, "--data", str(local_data_dir), "--device", "cuda",
             "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
             "--loss-space", "km", "--huber-beta", "0.5", "--seed", str(SEED),
             "--no-resume", *extra_scale_args, "--output-dir", str(exp_dir)],
            [sys.executable, "-B", str(entry_script), *prefix_args, "predict",
             "--model", model_id, "--data", str(local_data_dir), "--device", "cuda",
             "--batch-size", str(BATCH_SIZE), "--output-dir", str(exp_dir)],
        ]

    for idx, cmd in enumerate(cmds, 1):
        print(f"\n>>> [{idx}/{len(cmds)}] 执行: {' '.join(cmd)}")
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
    print(f"\n🎉 训练与预测全部完成，结果已持久化至 Google Drive: {exp_dir}")

目标入口脚本 : scripts/network_training/base_pack/scm/main.py
解析模型标识 : scm_resnet18_range
ℹ️ 当前 RUN_TRAINING=False（预览命令模式）。在 Colab GPU 准备好后将 RUN_TRAINING 设为 True 即可启动训练。

## 3. `MoE_learn` 阶段式持续学习与动态专家增长运行器（零原始数据重算）

`scripts/network_training/MoE_learn/demo.py`（`--train-h5` / `--test-h5`）及下方单元的 `load_prepared_data(effective_cfg, train_h5, test_h5)` 直接消费 **模块一** 预先生成好的逐窗去均值 `RBD_SCM` HDF5，完全无需 `Origindata/`：
- 一份 `"before_then_after"` HDF5 数据集即可直接切换运行 `before_then_after`、`before_then_after_range` 以及 `holdout_near` / `holdout_middle` / `holdout_far` 五种持续学习实验配置！

In [4]:
# Cell 4: 直接基于预生成 HDF5 运行 MoE_learn 持续学习（GPU）
from copy import deepcopy
import csv
from datetime import datetime
import tomllib

# --- MoE_learn 运行参数 ---
RUN_MOE_LEARN = False              # 在 Colab GPU 就绪且模块一生成好 MoE_learn HDF5 后设为 True
MOE_LEARN_CONFIG = "holdout_middle"  # 可选: "demo" | "before_then_after" | "before_then_after_range"
                                     #      | "holdout_near" | "holdout_middle" | "holdout_far"
OVERRIDE_EPOCHS = None             # 保持 None 使用 TOML 默认轮数；快速验证可设为整数如 5
# -------------------------

MOE_LEARN_WORKFLOW_TOMLS = {
    "demo": ("demo", PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_demo.toml"),
    "before_then_after": ("before_then_after", PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_before_then_after.toml"),
    "before_then_after_range": ("before_then_after", PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_before_then_after_range.toml"),
    "holdout_near": ("before_then_after", PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_holdout_near.toml"),
    "holdout_middle": ("before_then_after", PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_holdout_middle.toml"),
    "holdout_far": ("before_then_after", PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_holdout_far.toml"),
}

manifest_key, toml_path = MOE_LEARN_WORKFLOW_TOMLS[MOE_LEARN_CONFIG]
raw_moe_cfg = tomllib.loads(toml_path.read_text(encoding="utf-8"))
print(f"已选择 MoE_learn 配置 : {toml_path.relative_to(PROJECT_ROOT)}")
print(f"所需预生成数据集键    : moe_learn['{manifest_key}'] (stage_mode={raw_moe_cfg['dataset']['stage_mode']}, stages={raw_moe_cfg['continual']['stages']})")

if not RUN_MOE_LEARN:
    print("ℹ️ 当前 RUN_MOE_LEARN=False（配置预览模式）。")
else:
    if not CUDA_AVAILABLE:
        raise RuntimeError("MoE_learn 需要 CUDA GPU：请先切换 Colab 运行时为 GPU！")
    if manifest_key not in MANIFEST.get("moe_learn", {}):
        raise KeyError(
            f"dataset_manifest.json 中缺少 moe_learn[{manifest_key!r}]，请先在模块一的 MOE_LEARN_MODES 中启用并同步。"
        )

    from scripts.network_training.MoE_learn.common.checkpoints import load_checkpoint
    from scripts.network_training.MoE_learn.common.training import require_cuda, run_stages
    from scripts.network_training.MoE_learn.config import validate_runtime_config
    from scripts.network_training.MoE_learn.data import load_prepared_data
    from scripts.network_training.MoE_learn.visualization.plot_stages import plot_stages

    # 1) 将预生成的 RBD_SCM_train.h5 / RBD_SCM_test.h5 拉到本地 SSD
    rel_dir = Path(MANIFEST["moe_learn"][manifest_key]["rel_dir"])
    local_ml_dir = LOCAL_DATASET_CACHE / rel_dir
    local_ml_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_DATASET_ROOT / rel_dir, local_ml_dir, dirs_exist_ok=True)

    # 2) 校验配置并在本地加载阶段数据（不触发任何 Origindata 读取）
    effective_cfg = deepcopy(validate_runtime_config(raw_moe_cfg))
    if OVERRIDE_EPOCHS is not None:
        for k in ("initial_epochs", "adapt_epochs", "warmup_epochs"):
            effective_cfg["training"][k] = int(OVERRIDE_EPOCHS)

    device = require_cuda("cuda")
    prepared = load_prepared_data(
        effective_cfg,
        local_ml_dir / "RBD_SCM_train.h5",
        local_ml_dir / "RBD_SCM_test.h5",
    )

    stamp = datetime.now().strftime("%m%d_%H%M%S")
    out_dir = DRIVE_EXP_ROOT / f"MoE_learn_{MOE_LEARN_CONFIG}_{stamp}"
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "data-metadata.json").write_text(
        json.dumps(prepared["metadata"], ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )

    # 3) 运行各阶段适应/专家增长候选评估与部署保存
    result = run_stages(effective_cfg, prepared, out_dir, device=device)
    for row in result["stages"]:
        _, saved = load_checkpoint(out_dir / row["checkpoint"], device, expected_metadata=prepared["metadata"])
        assert saved["stage"] == row["stage"] and saved["model_state_sha256"] == row["deployed_state_sha256"]
        row["checkpoint_restore"] = "PASS"

    report = {
        "pack": "MoE_learn",
        "suite": "experiment",
        "status": "PASS",
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device),
        "effective_config": effective_cfg,
        **result,
    }
    report_path = out_dir / "demo-report.json"
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    with (out_dir / "stages.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "stage", "action", "experts", "new_training_rows",
                "cumulative_training_rows", "current_rmse_km", "historic_rmse_km",
                "test_mae_km", "test_rmse_km", "replay_rows", "novelty",
            ],
        )
        writer.writeheader()
        for row in result["stages"]:
            writer.writerow({
                "stage": row["stage"],
                "action": row["action"],
                "experts": row["experts"],
                "new_training_rows": row["new_training_rows"],
                "cumulative_training_rows": row["cumulative_training_rows"],
                "current_rmse_km": row["current"]["rmse_km"],
                "historic_rmse_km": None if row["historic"] is None else row["historic"]["rmse_km"],
                "test_mae_km": None if row["test"] is None else row["test"]["mae_km"],
                "test_rmse_km": None if row["test"] is None else row["test"]["rmse_km"],
                "replay_rows": row["replay_rows"],
                "novelty": row.get("novelty"),
            })

    plot_stages(report_path, out_dir / "figures")
    print(f"🎉 MoE_learn ({MOE_LEARN_CONFIG}) 完成，报告与阶段曲线已保存至: {out_dir}")

已选择 MoE_learn 配置 : scripts/remote_workflow/MoE_learn/haiqin_holdout_middle.toml
所需预生成数据集键    : moe_learn['before_then_after'] (stage_mode=distance_holdout, stages=3)
ℹ️ 当前 RUN_MOE_LEARN=False（配置预览模式）。

## 4. 实验结果汇总与对比可视化

自动扫描两类目录并汇总关键指标：
1. 本 Notebook 在 Google Drive `DRIVE_EXP_ROOT` 下新产出的全部实验目录（直接解析预测 CSV 与 `stages.csv` 计算 `MAE / RMSE / R²`，即使在未携带 `output/data/` 的 Colab 克隆环境中也可独立汇总与出图）；
2. 仓库本地 `output/data/` 中已归档的基准实验报告（若存在则自动叠加展示）。

In [5]:
# Cell 5: 扫描 Google Drive 新实验结果与仓库归档结果 (output/data/)
import csv
import math


def compute_csv_metrics(pred_csv: Path) -> dict[str, float] | None:
    with pred_csv.open(encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    if not rows:
        return None
    preds = [float(r["predicted_range_km"]) for r in rows]
    trues = [float(r["true_range_km"]) for r in rows]
    n = len(rows)
    mae = sum(abs(p - t) for p, t in zip(preds, trues)) / n
    rmse = math.sqrt(sum((p - t) ** 2 for p, t in zip(preds, trues)) / n)
    mean_t = sum(trues) / n
    ss_tot = sum((t - mean_t) ** 2 for t in trues)
    r2 = 1.0 - (sum((p - t) ** 2 for p, t in zip(preds, trues)) / ss_tot) if ss_tot > 0 else float("nan")
    return {"count": n, "mae_km": mae, "rmse_km": rmse, "r2": r2}


# 1) 扫描 Google Drive 实验目录中的新结果
DRIVE_RUN_METRICS: list[dict] = []
DRIVE_MOE_STAGES: list[tuple[str, list[dict]]] = []
if DRIVE_EXP_ROOT.exists():
    drive_runs = sorted([p for p in DRIVE_EXP_ROOT.iterdir() if p.is_dir()])
else:
    drive_runs = []

print(f"=== Google Drive 实验产出目录 ({DRIVE_EXP_ROOT}) : 共 {len(drive_runs)} 个 ===")
for run_dir in drive_runs:
    stage_csv = run_dir / "stages.csv"
    if stage_csv.is_file():
        with stage_csv.open(encoding="utf-8") as fh:
            st_rows = list(csv.DictReader(fh))
        if st_rows:
            DRIVE_MOE_STAGES.append((run_dir.name, st_rows))
            last = st_rows[-1]
            print(
                f"- [MoE_learn] {run_dir.name} | 最终 Stage {last['stage']} ({last['action']}, E{last['experts']}) "
                f"| Test RMSE={float(last['test_rmse_km']):.4f} km"
            )
        continue
    pred_csvs = sorted(run_dir.rglob("*_predictions_*.csv"))
    if pred_csvs:
        m = compute_csv_metrics(pred_csvs[-1])
        if m:
            DRIVE_RUN_METRICS.append({"run": run_dir.name, **m})
            print(
                f"- {run_dir.name} | RMSE={m['rmse_km']:.4f} km | MAE={m['mae_km']:.4f} km | R2={m['r2']:.4f} (N={m['count']})"
            )
    else:
        print(f"- {run_dir.name} (进行中或无测试集预测 CSV)")

# 2) 读取仓库归档基准数据（若存在 output/data/）供对照参考
OUTPUT_DATA = PROJECT_ROOT / "output" / "data"
ARCHIVE_SUMMARY: dict[str, list] = {}

base_csv = OUTPUT_DATA / "20260417T060103Z-base-haiqin_range_overlap_train_before-1b5676c1" / "comparison" / "model_metrics.csv"
if base_csv.exists():
    with base_csv.open(encoding="utf-8") as f:
        rows = sorted(csv.DictReader(f), key=lambda r: float(r["rmse_km"]))
    ARCHIVE_SUMMARY["base_pack"] = rows
    print("\n[基准参考] base_pack (train_before) Top-5 测试集 RMSE:")
    for r in rows[:5]:
        print(f"  {r['method']:4s} | {r['network']:12s} | RMSE={float(r['rmse_km']):.4f} km | MAE={float(r['mae_km']):.4f} km | R2={float(r['r2']):.4f}")

joint_csv = OUTPUT_DATA / "20260417T060103Z-joint-haiqin_range_overlap_train_before-eb63d6cc" / "comparison" / "model_metrics.csv"
if joint_csv.exists():
    with joint_csv.open(encoding="utf-8") as f:
        j_rows = sorted(csv.DictReader(f), key=lambda r: float(r["rmse_km"]))
    ARCHIVE_SUMMARY["joint_feature_pack"] = j_rows
    print("\n[基准参考] joint_feature_pack (block_rms, train_before) Top-5 测试集 RMSE:")
    for r in j_rows[:5]:
        print(f"  {r['method']:11s} | {r['network']:12s} | RMSE={float(r['rmse_km']):.4f} km | MAE={float(r['mae_km']):.4f} km | R2={float(r['r2']):.4f}")

moe_csv = OUTPUT_DATA / "rbd-scm-moe-expert-study-v1-20260419T084805Z-89f97e8e" / "analysis" / "condition-summary.csv"
if moe_csv.exists():
    with moe_csv.open(encoding="utf-8") as f:
        m_rows = list(csv.DictReader(f))
    ARCHIVE_SUMMARY["MoE_data_scale"] = m_rows

holdout_dirs = {
    "near": "20260420T090427Z-moelearn-haiqin_holdout_near-5ec04519",
    "middle": "20260420T090427Z-moelearn-haiqin_holdout_middle-e1cd410e",
    "far": "20260420T090428Z-moelearn-haiqin_holdout_far-94b211a8",
}
ARCHIVE_SUMMARY["MoE_learn_holdout"] = []
for group_name, folder_name in holdout_dirs.items():
    st_path = OUTPUT_DATA / folder_name / "stages.csv"
    if st_path.exists():
        with st_path.open(encoding="utf-8") as f:
            st_rows = list(csv.DictReader(f))
        ARCHIVE_SUMMARY["MoE_learn_holdout"].append((group_name, st_rows))

if ARCHIVE_SUMMARY["MoE_learn_holdout"]:
    print("\n[基准参考] MoE_learn 距离留出持续学习 (Stage 1 -> Stage 3 测试 RMSE):")
    for group_name, st_rows in ARCHIVE_SUMMARY["MoE_learn_holdout"]:
        s1, s3 = st_rows[0], st_rows[-1]
        print(
            f"  holdout={group_name:6s} | Stage1({s1['action']}, E{s1['experts']}): {float(s1['test_rmse_km']):.4f} km "
            f"-> Stage3({s3['action']}, E{s3['experts']}): {float(s3['test_rmse_km']):.4f} km"
        )

=== Google Drive 实验产出目录 (/Volumes/exDateDisk/projects/SwellEx/outputs/drive_staging/experiments) : 共 0 个 ===

In [6]:
# Cell 6: 绘制实验对比看板（支持 Google Drive 新产出实验或仓库归档基准）
has_plot_data = bool(DRIVE_RUN_METRICS or DRIVE_MOE_STAGES or ARCHIVE_SUMMARY.get("base_pack"))
if importlib.util.find_spec("matplotlib") is None or not has_plot_data:
    print("ℹ️ 未检测到 matplotlib 或当前尚无已完成的实验/归档数据，跳过图形渲染（完成上方训练后重新运行本单元即可出图）。")
else:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

    # (a) Google Drive 新实验 RMSE 或 Base Pack Top-8
    ax = axes[0, 0]
    if DRIVE_RUN_METRICS:
        sorted_drive = sorted(DRIVE_RUN_METRICS, key=lambda r: r["rmse_km"])[:10]
        labels = [r["run"][:32] for r in sorted_drive][::-1]
        vals = [r["rmse_km"] for r in sorted_drive][::-1]
        ax.barh(labels, vals, color="#2b6cb0")
        ax.set_title("Google Drive New Runs Test RMSE (km)")
    elif ARCHIVE_SUMMARY.get("base_pack"):
        top_base = ARCHIVE_SUMMARY["base_pack"][:8]
        labels = [f"{r['method']}-{r['network']}" for r in top_base][::-1]
        vals = [float(r["rmse_km"]) for r in top_base][::-1]
        ax.barh(labels, vals, color="#2b6cb0")
        ax.set_title("Base Pack Top-8 Test RMSE (km)")
    else:
        ax.text(0.5, 0.5, "No standard run metrics yet", ha="center", va="center")
        ax.set_title("Standard Runs Test RMSE (km)")
    ax.set_xlabel("RMSE (km)")

    # (b) Joint Feature Pack Top-8 (若存在归档) 或新实验 MAE vs RMSE
    ax = axes[0, 1]
    top_joint = ARCHIVE_SUMMARY.get("joint_feature_pack", [])[:8]
    if top_joint:
        j_labels = [f"{r['method']}-{r['network']}" for r in top_joint][::-1]
        j_vals = [float(r["rmse_km"]) for r in top_joint][::-1]
        ax.barh(j_labels, j_vals, color="#2f855a")
        ax.set_title("Joint Feature Pack (block_rms) Top-8 RMSE (km)")
        ax.set_xlabel("RMSE (km)")
    elif DRIVE_RUN_METRICS:
        sorted_drive = sorted(DRIVE_RUN_METRICS, key=lambda r: r["mae_km"])[:10]
        labels = [r["run"][:32] for r in sorted_drive][::-1]
        vals = [r["mae_km"] for r in sorted_drive][::-1]
        ax.barh(labels, vals, color="#2f855a")
        ax.set_title("Google Drive New Runs Test MAE (km)")
        ax.set_xlabel("MAE (km)")
    else:
        ax.text(0.5, 0.5, "Run standard/joint pack to populate", ha="center", va="center")
        ax.set_title("Joint / MAE Comparison")

    # (c) MoE_data_scale RBD_SCM Expert Matrix
    ax = axes[1, 0]
    moe_rows = ARCHIVE_SUMMARY.get("MoE_data_scale", [])
    if moe_rows:
        for e_tag in ("E1", "E2", "E4", "E8"):
            pts = sorted(
                [(int(float(r["n_train"])), float(r["test_rmse_km_mean"])) for r in moe_rows if r.get("expert_arm") == e_tag]
            )
            if pts:
                ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=e_tag)
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No MoE_data_scale matrix archive", ha="center", va="center")
    ax.set_title("MoE_data_scale RBD_SCM: N_train vs Test RMSE")
    ax.set_xlabel("Optimizer Training Samples (N_train)")
    ax.set_ylabel("Mean Test RMSE (km)")

    # (d) MoE_learn Stages (优先叠加 Google Drive 新跑出的持续学习曲线)
    ax = axes[1, 1]
    plotted_stages = False
    for run_label, st_rows in DRIVE_MOE_STAGES[-5:]:
        stages_x = [int(r["stage"]) for r in st_rows if r.get("test_rmse_km")]
        rmses_y = [float(r["test_rmse_km"]) for r in st_rows if r.get("test_rmse_km")]
        if stages_x:
            ax.plot(stages_x, rmses_y, marker="o", linewidth=2, label=run_label[:26])
            plotted_stages = True
    for group_name, st_rows in ARCHIVE_SUMMARY.get("MoE_learn_holdout", []):
        stages_x = [int(r["stage"]) for r in st_rows]
        rmses_y = [float(r["test_rmse_km"]) for r in st_rows]
        ax.plot(stages_x, rmses_y, marker="s", linestyle="--", label=f"archive_{group_name}")
        plotted_stages = True
    ax.set_title("MoE_learn Stage Progression")
    ax.set_xlabel("Continual Stage")
    ax.set_ylabel("Test RMSE (km)")
    if plotted_stages:
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "Run Cell 4 (MoE_learn) to plot stages", ha="center", va="center")

    if IN_COLAB or DRIVE_EXP_ROOT.exists():
        DRIVE_EXP_ROOT.mkdir(parents=True, exist_ok=True)
        fig_out = DRIVE_EXP_ROOT / "swellex_benchmark_overview.png"
        fig.savefig(fig_out, dpi=160)
        print(f"📊 对比图已保存至: {fig_out}")
    plt.show()

ℹ️ 未检测到 matplotlib 或当前尚无已完成的实验/归档数据，跳过图形渲染（完成上方训练后重新运行本单元即可出图）。